# MCP Proxy Analysis - Google Colab (GPU)

Questo notebook esegue l'analisi proxy LLM su **tutti i server** dell'Excel, sfruttando la **GPU** di Colab per accelerare l'inferenza di Ollama (llama3).

**Requisiti**: Runtime con GPU (**Runtime > Change runtime type > T4 GPU**)

**Nessun file del progetto viene modificato.** Il notebook usa il codice originale della Pipeline.

## 0. Verifica GPU

In [ ]:
!nvidia-smi

## 1. Installa Ollama e avvia il server

In [ ]:
# Installa zstd (necessario per estrarre Ollama) e poi Ollama
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess
import time
import os

# Avvia ollama serve in background
ollama_proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(5)
print(f'Ollama server avviato (PID: {ollama_proc.pid})')

In [ ]:
# Scarica il modello llama3 (usa la GPU automaticamente)
!ollama pull llama3

In [ ]:
# Verifica che Ollama funzioni
!echo 'Rispondi solo: OK' | ollama run llama3

## 2. Installa Node.js, Go e dipendenze

In [ ]:
# Installa Node.js 20 LTS + Go
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y nodejs golang-go
!node --version && npm --version && go version

In [ ]:
# Installa ts-node e typescript globalmente
!npm install -g ts-node typescript

In [ ]:
# Installa dipendenze Python
!pip install -q pandas openpyxl

## 3. Carica il progetto Pipeline

In [ ]:
import os

REPO_URL = 'https://github.com/frax01/pipeline.git'  # <-- MODIFICA con il tuo URL
PIPELINE_DIR = '/content/Pipeline'

if not os.path.exists(PIPELINE_DIR):
    !git clone {REPO_URL} {PIPELINE_DIR}
else:
    !cd {PIPELINE_DIR} && git pull

print(f'Pipeline directory: {PIPELINE_DIR}')

In [ ]:
import os

# Setup symlinks: Path.home()/Desktop/Pipeline -> /content/Pipeline
# Necessario perche' config.py usa Path.home() / "Desktop/Pipeline"
!mkdir -p ~/Desktop
!ln -sfn /content/Pipeline ~/Desktop/Pipeline

# Crea directory necessarie
!mkdir -p /content/Pipeline/analysis
!test -f /content/Pipeline/claude_desktop_config.json || echo '{}' > /content/Pipeline/claude_desktop_config.json

# Installa dipendenze npm del progetto llmProxy
!cd /content/Pipeline/frameworks/llmProxy && npm install

# Rendi eseguibile lo script di build
!chmod +x /content/Pipeline/npm_runner/npm_build.sh

# Crea wrapper per /opt/homebrew/bin/npx
# llmProxyAnalysis.py chiama "/opt/homebrew/bin/npx ts-node index.ts ..."
# ts-node non funziona su Colab con ESM, ma dist/ ha gia i JS compilati
# Il wrapper intercetta "npx ts-node file.ts" -> "node dist/file.js"
os.makedirs('/opt/homebrew/bin', exist_ok=True)
with open('/opt/homebrew/bin/npx', 'w') as f:
    f.write('#!/bin/bash\n')
    f.write('if [ "$1" = "ts-node" ]; then\n')
    f.write('    shift\n')
    f.write('    TS_FILE="$1"\n')
    f.write('    shift\n')
    f.write('    DIR=$(dirname "$TS_FILE")\n')
    f.write('    BASE=$(basename "$TS_FILE" .ts)\n')
    f.write('    JS_FILE="$DIR/dist/$BASE.js"\n')
    f.write('    exec node "$JS_FILE" "$@"\n')
    f.write('else\n')
    f.write('    exec /usr/bin/npx "$@"\n')
    f.write('fi\n')
!chmod +x /opt/homebrew/bin/npx

# Verifica che il wrapper funzioni
!cat /opt/homebrew/bin/npx
print('\nSetup completato')

## 4. Carica l'Excel con i server

Carica il file Excel con tutti i server. Deve avere la colonna `Link`.

In [ ]:
from google.colab import files
import pandas as pd

print('Carica il file Excel con i server (deve avere la colonna "Link"):')
uploaded = files.upload()

EXCEL_FILENAME = list(uploaded.keys())[0]
EXCEL_PATH = f'/content/{EXCEL_FILENAME}'

df_excel = pd.read_excel(EXCEL_PATH)
print(f'\nFile caricato: {EXCEL_FILENAME}')
print(f'Totale server: {len(df_excel)}')
print(f'Colonne: {list(df_excel.columns)}')
df_excel.head()

## 5. Configurazione analisi

In [ ]:
import sys
import json
from pathlib import Path

# Aggiungi Pipeline al sys.path per importare i moduli del progetto
if '/content/Pipeline' not in sys.path:
    sys.path.insert(0, '/content/Pipeline')

from functions.buildConfig import clone_repo, build_mcp_config
from functions.helper import detect_language, force_delete, extract_server_name
from frameworks.llmProxyAnalysis import run_llm_proxy_analysis

# ============================================
# CONFIGURAZIONE - MODIFICA QUI
# ============================================
START_IDX = 0          # Indice di partenza (0 = dall'inizio, modifica per riprendere)
END_IDX = None         # None = fino alla fine, oppure un numero (es. 100 per test)

# Directory di lavoro
WORK_DIR = Path('/content/Pipeline')

# File di output
RESULTS_FILE = Path('/content/proxy_results.json')
SERVERS_LOG_FILE = Path('/content/proxy_servers_log.json')
STATS_FILE = Path('/content/proxy_stats.json')

end_idx = END_IDX or len(df_excel)
print(f'Range: [{START_IDX} : {end_idx}] ({end_idx - START_IDX} server)')

## 6. Loop principale - Analisi di tutti i server

In [ ]:
import time
import traceback

# ============================================
# STRUTTURA STATISTICHE AGGREGATE
# ============================================
def empty_det_causes():
    return {
        "total": 0, "command_injection": 0, "ssh_private_keys": 0,
        "suspicious_file_access": 0, "sql_injection": 0,
        "container_isolation_violation": 0, "entropy_secrets": 0, "pii_detection": 0
    }

def empty_prompt_det_causes():
    d = empty_det_causes()
    d.update({"prompt_injection": 0, "important_tag_injection": 0,
              "shadow_hijack": 0, "cross_origin_access": 0, "xss_injection": 0})
    return d

def new_aggregate():
    return {
        "total_servers": 0, "completed": 0, "failed": 0, "skipped": 0,
        "total_tools": 0, "total_trials": 0, "total_failed_trials": 0,
        "languages": {},
        "tool_blocked": {
            "total": 0,
            "benign": {"deterministic_causes": empty_det_causes(), "tool_call_causes": 0, "tool_response_causes": 0},
            "malicious": {"deterministic_causes": empty_det_causes(), "tool_call_causes": 0, "tool_response_causes": 0}
        },
        "prompt": {
            "total_prompt": 0, "total_blocked": 0, "total_allowed": 0,
            "deterministic_causes": empty_prompt_det_causes(),
            "tool_call_causes": 0, "tool_response_causes": 0
        }
    }

# Carica stato precedente o crea nuovo
if STATS_FILE.exists() and START_IDX > 0:
    aggregate_stats = json.loads(STATS_FILE.read_text())
    servers_log = json.loads(SERVERS_LOG_FILE.read_text()) if SERVERS_LOG_FILE.exists() else {}
    all_results = json.loads(RESULTS_FILE.read_text()) if RESULTS_FILE.exists() else {}
    print(f'Resume: {aggregate_stats["completed"]} server gia completati')
else:
    aggregate_stats = new_aggregate()
    servers_log = {}
    all_results = {}


def update_aggregate(agg, proxy_result, server_language, tool_length):
    proxy_data = proxy_result.get("proxy", {})
    prompt_data = proxy_result.get("prompt", {})

    agg["completed"] += 1
    agg["languages"][server_language] = agg["languages"].get(server_language, 0) + 1
    agg["total_tools"] += tool_length
    agg["total_trials"] += proxy_result.get("total_trials", 0)
    agg["total_failed_trials"] += proxy_result.get("total_failed", 0)

    tb = agg["tool_blocked"]
    for intent in ("benign", "malicious"):
        src = proxy_data.get(intent, {})
        src_det = src.get("deterministic_causes", {})
        dst_det = tb[intent]["deterministic_causes"]
        for key in dst_det:
            dst_det[key] += src_det.get(key, 0)
        tb[intent]["tool_call_causes"] += src.get("tool_call_causes", 0)
        tb[intent]["tool_response_causes"] += src.get("tool_response_causes", 0)
    tb["total"] = sum([
        tb["benign"]["deterministic_causes"]["total"],
        tb["malicious"]["deterministic_causes"]["total"],
        tb["benign"]["tool_call_causes"], tb["benign"]["tool_response_causes"],
        tb["malicious"]["tool_call_causes"], tb["malicious"]["tool_response_causes"]
    ])

    ap = agg["prompt"]
    ap["total_prompt"] += prompt_data.get("total_prompt", 0)
    ap["total_blocked"] += prompt_data.get("total_blocked", 0)
    ap["total_allowed"] += prompt_data.get("total_allowed", 0)
    ap["tool_call_causes"] += prompt_data.get("tool_call_causes", 0)
    ap["tool_response_causes"] += prompt_data.get("tool_response_causes", 0)
    for key, value in prompt_data.get("deterministic_causes", {}).items():
        if key in ap["deterministic_causes"]:
            ap["deterministic_causes"][key] += value


def save_progress():
    STATS_FILE.write_text(json.dumps(aggregate_stats, indent=2))
    SERVERS_LOG_FILE.write_text(json.dumps(servers_log, indent=2))
    RESULTS_FILE.write_text(json.dumps(all_results, indent=2))


# ============================================
# LOOP PRINCIPALE
# ============================================
total_to_process = end_idx - START_IDX
print(f'Inizio analisi: server {START_IDX} -> {end_idx} ({total_to_process} server)')
print('=' * 70)

for idx, row in df_excel.iloc[START_IDX:end_idx].iterrows():
    server_url = row["Link"]
    server_name = extract_server_name(server_url)
    start_time = time.time()
    aggregate_stats["total_servers"] += 1
    progress = aggregate_stats["total_servers"]

    print(f'\n{"#" * 60}')
    print(f'[{progress}/{total_to_process}] Index: {idx} | {server_name}')
    print(f'URL: {server_url}')

    repo_path = None

    try:
        # 1. Clone
        repo_path = clone_repo(server_url, WORK_DIR)

        # 2. Detect language
        server_language = detect_language(repo_path)
        print(f'Language: {server_language}')

        # 3. Build config
        _, command, elem = build_mcp_config(repo_path, server_language)

        if command is None or elem is None:
            print(f'[SKIP] Server non eseguibile')
            aggregate_stats["skipped"] += 1
            servers_log[server_name] = {
                "url": server_url, "index": idx,
                "language": server_language, "status": "skipped"
            }
            save_progress()
            continue

        print(f'Command: {command} {" ".join(str(e) for e in elem)}')

        # 4. Proxy analysis
        print(f'[PROXY] Avvio analisi...')
        proxy_result = run_llm_proxy_analysis(repo_path, command, elem)
        status = proxy_result.get("status", "failed")
        print(f'[PROXY] Status: {status}')

        if status == "completed":
            tool_length = proxy_result.get("tool_length", 0)
            update_aggregate(aggregate_stats, proxy_result, server_language, tool_length)
            all_results[server_name] = proxy_result
            servers_log[server_name] = {
                "url": server_url, "index": idx,
                "language": server_language, "status": "completed",
                "tool_length": tool_length,
                "total_trials": proxy_result.get("total_trials", 0)
            }
        else:
            aggregate_stats["failed"] += 1
            servers_log[server_name] = {
                "url": server_url, "index": idx,
                "language": server_language, "status": "failed"
            }

    except Exception as e:
        print(f'[ERRORE] {e}')
        traceback.print_exc()
        aggregate_stats["failed"] += 1
        servers_log[server_name] = {
            "url": server_url, "index": idx,
            "status": "error", "error": str(e)
        }
    finally:
        # 5. Pulizia repo clonato
        if repo_path and repo_path.exists() and repo_path.is_dir():
            try:
                force_delete(repo_path)
            except Exception:
                pass

        # 6. Salva progresso dopo ogni server
        save_progress()

        elapsed = time.time() - start_time
        c = aggregate_stats
        print(f'Tempo: {elapsed:.1f}s | Completati: {c["completed"]} | '
              f'Failed: {c["failed"]} | Skipped: {c["skipped"]}')

print(f'\n{"=" * 70}')
print('ANALISI COMPLETATA')
c = aggregate_stats
print(f'Totale: {c["total_servers"]} | Completati: {c["completed"]} | '
      f'Failed: {c["failed"]} | Skipped: {c["skipped"]}')

## 7. Statistiche finali

In [ ]:
stats = json.loads(Path('/content/proxy_stats.json').read_text())

print('=' * 60)
print('STATISTICHE AGGREGATE')
print('=' * 60)
print(f'Server processati: {stats["total_servers"]}')
print(f'  Completati: {stats["completed"]}')
print(f'  Falliti:    {stats["failed"]}')
print(f'  Skippati:   {stats["skipped"]}')
print(f'Linguaggi: {json.dumps(stats["languages"], indent=2)}')
print(f'\nTool analizzati: {stats["total_tools"]}')
print(f'Trial totali: {stats["total_trials"]}')
print(f'Trial falliti: {stats["total_failed_trials"]}')

tb = stats["tool_blocked"]
print(f'\n--- TOOL BLOCKED (totale: {tb["total"]}) ---')
for intent in ("benign", "malicious"):
    det = tb[intent]["deterministic_causes"]
    print(f'  [{intent.upper()}] det:{det["total"]} | llm_call:{tb[intent]["tool_call_causes"]} | llm_resp:{tb[intent]["tool_response_causes"]}')
    for k, v in det.items():
        if k != "total" and v > 0:
            print(f'    {k}: {v}')

p = stats["prompt"]
pct = f'{p["total_blocked"]/p["total_prompt"]*100:.1f}%' if p["total_prompt"] > 0 else 'N/A'
print(f'\n--- PROMPT (totale: {p["total_prompt"]}, bloccati: {p["total_blocked"]} = {pct}) ---')
print(f'  Consentiti: {p["total_allowed"]}')
print(f'  LLM call: {p["tool_call_causes"]} | LLM resp: {p["tool_response_causes"]}')
pdet = p["deterministic_causes"]
for k, v in pdet.items():
    if k != "total" and v > 0:
        print(f'  {k}: {v}')

## 8. Scarica i risultati

In [ ]:
from google.colab import files

for f in ['/content/proxy_stats.json', '/content/proxy_servers_log.json', '/content/proxy_results.json']:
    if os.path.exists(f):
        files.download(f)
        print(f'Scaricato: {f}')

## 9. Riprendere l'analisi dopo un'interruzione

Se Colab si disconnette o l'analisi si interrompe:
1. I risultati parziali sono salvati dopo ogni server
2. Ri-carica l'Excel (cella 4)
3. Modifica `START_IDX` nella cella 5 con l'indice suggerito qui sotto
4. Ri-esegui dalla cella 5 in poi

In [ ]:
# Verifica ultimo indice processato
if os.path.exists('/content/proxy_servers_log.json'):
    log = json.loads(Path('/content/proxy_servers_log.json').read_text())
    if log:
        last_idx = max(v.get('index', 0) for v in log.values())
        print(f'Ultimo indice processato: {last_idx}')
        print(f'Per riprendere, imposta START_IDX = {last_idx + 1}')
        print(f'Server processati finora: {len(log)}')
    else:
        print('Nessun server processato ancora')
else:
    print('Nessun log trovato')

## 10. (Opzionale) Modello piu' grande

Con la GPU di Colab puoi provare modelli piu' potenti per risultati di verifica migliori.

In [ ]:
# Per cambiare modello, modifica config.ts nel repo clonato e pull il modello:
#
# T4 (16GB VRAM):
#   llama3      (default, ~5GB)    mistral (~5GB)
#   gemma2:9b   (~6GB)             qwen2.5:14b (~9GB)
#
# A100 (40/80GB VRAM):
#   llama3:70b  (~40GB)
#
# !sed -i "s/export const OLLAMA_MODEL = 'llama3'/export const OLLAMA_MODEL = 'mistral'/" /content/Pipeline/frameworks/llmProxy/config.ts
# !ollama pull mistral

tecnico@carminati-martignoni-1:~$ echo '=== VM1 guard ===' && free -h && echo && df -h /home

=== VM1 guard ===

               total        used        free      shared  buff/cache   available

Mem:            15Gi       1.3Gi       1.4Gi       144Mi        13Gi        14Gi

Swap:             0B          0B          0B

Filesystem      Size  Used Avail Use% Mounted on

/dev/vda1        96G   96G     0 100% /

tecnico@carminati-martignoni-1:~$



la memoria di guard ad esempio è piena, forse perchè non elimina i repo oppure non libera la memoria come invece dovrebbe fare. Come faccio a risolvere questo problema per non far bloccare l'esecuzione? Questo succede praticamente con tutti i tool delle diverse vm